# WISE² RECAST ENGINE × OpenJutsu — Free Kaggle GPU Backend (T4 x2 · 30 GB RAM)

**Why Kaggle?**
- **30 hours/week FREE GPU quota**.
- **Dual NVIDIA T4 (2 × 16 GB VRAM) + 30 GB System RAM** — significantly better RAM headroom than free Colab for MiniMax H3 INT8 + Qwen3-VL 4B/8B.
- Enable **Accelerator: GPU T4 x2** and **Internet: ON** in Kaggle Notebook Settings.

In [ ]:
!nvidia-smi
!free -h
!df -h /kaggle/working

In [ ]:
import os, subprocess
WORK = '/kaggle/working/ComfyUI'
if not os.path.exists(WORK):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Comfy-Org/ComfyUI.git', WORK], check=True)
subprocess.run(['pip', 'install', '-q', '-r', f'{WORK}/requirements.txt'], check=True)

CUSTOM_NODES = [
    'https://github.com/D3voz/comfyui-h3-oneclick-swap.git',
    'https://github.com/D3voz/comfyui-websocket-reconnect-fix.git',
    'https://github.com/D3voz/comfyui-image-detail-contrast.git',
    'https://github.com/ethanfel/ComfyUI-MiniMaxH3-Context-Loop.git',
    'https://github.com/Adudeguyman/ComfyUI-Fantastic-MiniMaxH3-PromptBuilder.git',
    'https://github.com/kijai/ComfyUI-KJNodes.git',
    'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git',
    'https://github.com/orion4d/Orion4D_FXMax.git',
]
os.makedirs(f'{WORK}/custom_nodes', exist_ok=True)
for repo in CUSTOM_NODES:
    name = repo.rstrip('/').split('/')[-1].replace('.git', '')
    target = f'{WORK}/custom_nodes/{name}'
    if not os.path.exists(target):
        subprocess.run(['git', 'clone', '--depth', '1', repo, target], check=True)
    req = f'{target}/requirements.txt'
    if os.path.exists(req):
        subprocess.run(['pip', 'install', '-q', '-r', req], check=False)
print('Kaggle ComfyUI + OpenJutsu nodes ready.')

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!chmod +x /tmp/cloudflared

import subprocess, time, re
comfy = subprocess.Popen(['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188'], cwd='/kaggle/working/ComfyUI')
time.sleep(8)
tunnel = subprocess.Popen(['/tmp/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        print('\n====================================================')
        print('PASTE THIS URL INTO WISE2 RECAST -> GPU BACKENDS -> KAGGLE GPU:')
        print(m.group(0))
        print('====================================================\n')
        break
comfy.wait()